In [21]:
import os
import pickle
import numpy as np
import shutil

def aggregate_raw_gradients():
    base_folder = "received_chunks_bin"
    if not os.path.exists(base_folder):
        print("[SERVER] No received folder found.")
        return

    client_dirs = [d for d in os.listdir(base_folder) if os.path.isdir(os.path.join(base_folder, d))]

    if not client_dirs:
        print("[SERVER] No client gradients to aggregate.")
        return

    # Step 1: Load raw chunks
    chunk_groups = dict()

    for client_id in client_dirs:
        client_path = os.path.join(base_folder, client_id)
        chunk_files = sorted(
            [f for f in os.listdir(client_path) if f.endswith(".bin")],
            key=lambda x: int(x.split("_")[-1].split(".")[0])
        )

        for fname in chunk_files:
            try:
                idx = int(fname.split("_")[-1].split(".")[0])
                with open(os.path.join(client_path, fname), "rb") as f:
                    payload = pickle.load(f)
                    if isinstance(payload, dict) and "data" in payload:
                        vec = np.array(payload["data"])  # now raw numpy array
                        chunk_groups.setdefault(idx, []).append(vec)
                        print(f"[SERVER] Loaded chunk: {fname}")
                    else:
                        print(f"[SERVER] Invalid format in {fname}")
            except Exception as e:
                print(f"[SERVER] Failed to load {fname}: {e}")

    if not chunk_groups:
        print("[SERVER] No valid chunks to aggregate.")
        return

    # Step 2: Aggregate each chunk
    aggregated_chunks = []

    for idx, vectors in sorted(chunk_groups.items()):
        try:
            stacked = np.stack(vectors)
            avg = np.mean(stacked, axis=0)
            aggregated_chunks.append((idx, avg))
            print(f"[SERVER] Aggregated chunk index {idx}")
        except Exception as e:
            print(f"[SERVER] Failed to aggregate chunk {idx}: {e}")

    # Step 3: Save aggregated chunks
    output_folder = "aggregated_chunks_bin"
    os.makedirs(output_folder, exist_ok=True)

    for idx, chunk in aggregated_chunks:
        with open(os.path.join(output_folder, f"agg_chunk_{idx}.bin"), "wb") as f:
            pickle.dump(chunk, f)

    print(f"[SERVER] Aggregated {len(aggregated_chunks)} raw chunks saved in '{output_folder}'")

    # Step 4: Combine all chunks into a global file
    all_arrays = [chunk for _, chunk in sorted(aggregated_chunks)]
    with open("aggregated_gradient_global_raw.pkl", "wb") as f:
        pickle.dump(all_arrays, f)
    print("[SERVER] Final global gradient ready to be sent to clients.")

    # Cleanup
    try:
        shutil.rmtree(base_folder)
        print("[SERVER] Cleaned up 'received_chunks_bin' for next round.")
    except Exception as e:
        print(f"[SERVER] Failed to clean up: {e}")

if __name__ == "__main__":
    aggregate_raw_gradients()


[SERVER] Loaded chunk: chunk_0.bin
[SERVER] Loaded chunk: chunk_1.bin
[SERVER] Aggregated chunk index 0
[SERVER] Aggregated chunk index 1
[SERVER] Aggregated 2 raw chunks saved in 'aggregated_chunks_bin'
[SERVER] Final global gradient ready to be sent to clients.
[SERVER] Cleaned up 'received_chunks_bin' for next round.
